[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jgcalvo/SP1306/blob/main/edps-elipticas/MEF.ipynb)

# Elementos finitos P1 para la ecuación de Poisson

Traducción de `MEF_v1.m` y `MEF_v2.m`.

$$-\Delta u = f \quad \text{en } \Omega=(0,1)\times(0,1), \qquad u = 0 \ \text{ sobre } \partial\Omega$$

**Formulación variacional.** Multiplicando por una función de prueba $v$ que se
anula en la frontera e integrando por partes,

$$\int_\Omega \nabla u \cdot \nabla v = \int_\Omega f\, v \qquad \forall v$$

Se aproxima $u$ por una función **continua y lineal a trozos** sobre una
triangulación. Tomando como base las funciones sombrero $\varphi_i$ (que valen 1
en el nodo $i$ y 0 en los demás), el problema se vuelve el sistema
$A\mathbf{u} = \mathbf{b}$ con

$$A_{ij} = \int_\Omega \nabla\varphi_i \cdot \nabla\varphi_j, \qquad b_i = \int_\Omega f\,\varphi_i$$

$A$ se llama **matriz de rigidez**. Como $\varphi_i$ solo es distinta de cero en
los triángulos que tocan al nodo $i$, casi todas las entradas son nulas.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import coo_matrix
from scipy.sparse.linalg import spsolve

## Entradas

In [ ]:
f   = lambda x, y: 2*np.pi**2*np.sin(np.pi*x)*np.sin(np.pi*y)
uex = lambda x, y: np.sin(np.pi*x)*np.sin(np.pi*y)

## 1. La malla

`node` guarda las coordenadas de los nodos (una fila por nodo) y `elem` la
conectividad: la fila $t$ tiene los tres índices de los vértices del triángulo
$t$, en sentido antihorario.

Cada celda cuadrada de la malla se parte en dos triángulos. Se conserva la
numeración de los `.m`: el nodo $(x_i, y_j)$ ocupa la posición
$i(M+1) + j$ — el índice $j$ (dirección $y$) es el rápido.

Nota: en NumPy los índices arrancan en 0, así que `elem` guarda índices
0-based. En MATLAB los mismos números aparecen corridos en 1.

In [ ]:
def squaremesh_basico(M):
    """Malla estructurada de (0,1)^2 con M subintervalos por lado.

    node: ((M+1)^2, 2)   elem: (2*M^2, 3), antihorarios."""
    xx = np.linspace(0, 1, M+1)
    X, Y = np.meshgrid(xx, xx, indexing='ij')   # X[i,j] = x_i,  Y[i,j] = y_j
    node = np.column_stack([X.ravel(), Y.ravel()])

    elem = np.zeros((2*M*M, 3), dtype=int)
    k = 0
    for i in range(M):                 # direccion x
        for j in range(M):             # direccion y
            n1 = i*(M+1) + j           # (x_i,     y_j)
            n2 = n1 + (M+1)            # (x_{i+1}, y_j)
            n3 = n2 + 1                # (x_{i+1}, y_{j+1})
            n4 = n1 + 1                # (x_i,     y_{j+1})
            elem[k]   = [n1, n2, n3]   # antihorario
            elem[k+1] = [n1, n3, n4]   # antihorario
            k += 2
    return node, elem

Los nodos de la frontera se detectan por un criterio puramente topológico, sin
mirar coordenadas: **una arista es de frontera si pertenece a un solo
triángulo**. Ordenando los dos índices de cada arista, las interiores aparecen
dos veces y las de borde una.

In [ ]:
def nodos_frontera(elem):
    """Nodos que estan sobre la frontera de la triangulacion."""
    total = np.vstack([elem[:, [1, 2]], elem[:, [2, 0]], elem[:, [0, 1]]])
    total = np.sort(total, axis=1)                    # cada arista, ordenada
    aristas, conteo = np.unique(total, axis=0, return_counts=True)
    bd_edge = aristas[conteo == 1]                    # aparecen una sola vez
    return np.unique(bd_edge)

### Ver la malla

Con `n = 4` se pueden rotular los nodos y los elementos, que es la forma de
entender qué guarda `elem`. Para mallas finas esto es ilegible.

In [ ]:
def dibujar_malla(node, elem, etiquetas=True):
    fig, ax = plt.subplots(figsize=(5.5, 5.5))
    ax.triplot(node[:, 0], node[:, 1], elem, color='0.25', lw=0.8)
    ax.plot(node[:, 0], node[:, 1], 'k.', ms=5)
    if etiquetas:
        d = 0.02
        for i, (x, y) in enumerate(node):
            ax.text(x+d, y+d, str(i), color='b', fontsize=9)
        bc = node[elem].mean(axis=1)               # baricentros
        for t, (x, y) in enumerate(bc):
            ax.text(x, y, str(t), color='r', fontsize=9, ha='center', va='center')
    ax.set_aspect('equal')
    plt.show()

node, elem = squaremesh_basico(4)
print(f'{len(node)} nodos, {len(elem)} triangulos')
dibujar_malla(node, elem)

## 2. Ensamblaje, recorriendo los elementos

Es la versión de `MEF_v1.m`. Sobre cada triángulo $T$ se calculan las
contribuciones locales

$$A^T_{ij} = |T|\,(\nabla\varphi_i \cdot \nabla\varphi_j), \qquad
b^T_i = \int_T f\,\varphi_i \approx \frac{|T|}{3}\,f(\text{baricentro})$$

y se suman en las posiciones globales que indica `elem`. Los gradientes son
constantes en cada triángulo, así que la integral es solo "valor × área".

El lado derecho usa la regla del baricentro: un solo punto de cuadratura, exacta
para polinomios de grado 1, que es suficiente para no estropear el orden 2 del
método.

In [ ]:
def ensamblar_por_elementos(node, elem, f):
    """Recorre los NT elementos; matriz DENSA, como en MEF_v1.m."""
    NV, NT = len(node), len(elem)
    A = np.zeros((NV, NV))
    b = np.zeros(NV)

    for t in range(NT):
        idx = elem[t]                          # indices globales
        x, y = node[idx, 0], node[idx, 1]

        # area con signo (positiva porque la malla es antihoraria)
        area = 0.5*((x[1]-x[0])*(y[2]-y[0]) - (x[2]-x[0])*(y[1]-y[0]))

        # gradientes de las tres funciones de forma
        g = np.array([[y[1]-y[2], x[2]-x[1]],
                      [y[2]-y[0], x[0]-x[2]],
                      [y[0]-y[1], x[1]-x[0]]]) / (2*area)

        xb, yb = x.mean(), y.mean()            # baricentro
        for i in range(3):
            for j in range(3):
                A[idx[i], idx[j]] += area*(g[i] @ g[j])
            b[idx[i]] += area*f(xb, yb)/3
    return A, b

## 3. Resolver

Los nodos de frontera no son incógnitas: valen 0. Se resuelve solo el bloque de
los nodos libres.

In [ ]:
n = 32
node, elem = squaremesh_basico(n)
NV = len(node)

bd_node = nodos_frontera(elem)
free = np.setdiff1d(np.arange(NV), bd_node)

A, b = ensamblar_por_elementos(node, elem, f)

u = np.zeros(NV)
u[free] = np.linalg.solve(A[np.ix_(free, free)], b[free])

ue = uex(node[:, 0], node[:, 1])
print(f'nodos: {NV}   libres: {len(free)}   frontera: {len(bd_node)}')
print(f'error en norma infinito: {np.linalg.norm(ue - u, np.inf):.6e}')

In [ ]:
fig = plt.figure(figsize=(7, 5))
ax = fig.add_subplot(projection='3d')
ax.plot_trisurf(node[:, 0], node[:, 1], u, triangles=elem, cmap='viridis')
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('$u_h$')
plt.show()

## 4. Ensamblaje vectorizado

Es la versión de `MEF_v2.m`, que sigue `assembling.m` de las notas del curso.

**La idea.** En vez de recorrer los `NT` elementos por fuera y los 9 pares
$(i,j)$ por dentro, se invierte el orden: se recorren los **9 pares por fuera**,
y cada uno se calcula para todos los elementos de una sola vez. El ciclo tiene 9
iteraciones sin importar el tamaño de la malla.

**La identidad que lo permite.** Sea $T$ un triángulo de área $|T|$ y sea
$\mathbf{v}_i$ el vector del lado *opuesto* al vértice $i$. El gradiente de la
función de forma es ese lado rotado 90°:

$$\nabla\varphi_i = \frac{\mathrm{rot}_{90}(\mathbf{v}_i)}{2|T|}$$

Como una rotación no cambia los productos punto,

$$\int_T \nabla\varphi_i\cdot\nabla\varphi_j
= |T|\,\frac{\mathbf{v}_i\cdot\mathbf{v}_j}{4|T|^2}
= \frac{\mathbf{v}_i\cdot\mathbf{v}_j}{4|T|}$$

No hay que calcular gradientes ni invertir nada: solo restas de coordenadas y un
producto punto. Además el área se toma en valor absoluto, así que no depende de
la orientación de los triángulos — la versión por elementos usa el área con
signo y falla si la malla no está orientada en sentido antihorario.

**El formato triplete.** Las contribuciones se acumulan en tres listas
`ii, jj, ss` y `coo_matrix` arma la matriz rala de una vez. Un mismo par
(fila, columna) aparece varias veces, una por cada elemento que comparte esos dos
nodos: al convertir a `csr` scipy **suma** los repetidos, que es exactamente el
ensamblaje.

In [ ]:
def ensamblar_vectorizado(node, elem, f):
    """Estilo assembling.m: 9 iteraciones, matriz rala."""
    NV, NT = len(node), len(elem)

    # ve[:,:,i] = lado opuesto al vertice i, para todos los elementos a la vez
    ve = np.zeros((NT, 2, 3))
    ve[:, :, 0] = node[elem[:, 2]] - node[elem[:, 1]]
    ve[:, :, 1] = node[elem[:, 0]] - node[elem[:, 2]]
    ve[:, :, 2] = node[elem[:, 1]] - node[elem[:, 0]]

    area = 0.5*np.abs(-ve[:, 0, 2]*ve[:, 1, 1] + ve[:, 1, 2]*ve[:, 0, 1])

    ii, jj, ss = [], [], []
    for i in range(3):
        for j in range(3):
            ii.append(elem[:, i])
            jj.append(elem[:, j])
            ss.append(np.sum(ve[:, :, i]*ve[:, :, j], axis=1)/(4*area))
    ii = np.concatenate(ii); jj = np.concatenate(jj); ss = np.concatenate(ss)
    A = coo_matrix((ss, (ii, jj)), shape=(NV, NV)).tocsr()   # suma repetidos
    # algunas sumas dan cero exacto y scipy igual las guarda; MATLAB las descarta
    A.eliminate_zeros()

    # lado derecho: misma regla del baricentro, tambien vectorizada
    bar = node[elem].mean(axis=1)                    # baricentro de cada triangulo
    aporte = area*f(bar[:, 0], bar[:, 1])/3
    b = np.bincount(elem.ravel(order='F'), weights=np.tile(aporte, 3), minlength=NV)
    return A, b

### Comprobar que da lo mismo

Antes de comparar tiempos hay que verificar que las dos rutas producen la misma
matriz y el mismo lado derecho.

In [ ]:
for nn in (4, 8, 16):
    nd, el = squaremesh_basico(nn)
    A1, b1 = ensamblar_por_elementos(nd, el, f)
    A2, b2 = ensamblar_vectorizado(nd, el, f)
    dA = np.abs(A2.toarray() - A1).max()
    db = np.abs(b2 - b1).max()
    dens = A2.nnz/len(nd)**2
    print(f'n={nn:2d}   max|A2-A1| = {dA:.3e}   max|b2-b1| = {db:.3e}'
          f'   no nulos: {A2.nnz}/{len(nd)**2} ({100*dens:.1f}%)')

Coinciden hasta el redondeo. Y se ve el motivo de fondo para usar matrices
ralas: el porcentaje de entradas no nulas se desploma al refinar.

Un detalle de scipy: varias de esas sumas dan **cero exacto**, y `coo_matrix`
las guarda igual como entradas almacenadas. `A.eliminate_zeros()` las descarta,
que es lo que hace `sparse` de MATLAB por su cuenta. Sin esa llamada, `nnz` da
más grande y el porcentaje de arriba quedaría inflado.

In [ ]:
print(f'{"n":>5} {"NV":>7} {"por elementos":>16} {"vectorizado":>14} {"A densa pesaria":>17}')
for nn in (16, 32, 64):
    nd, el = squaremesh_basico(nn)
    t0 = time.perf_counter(); ensamblar_por_elementos(nd, el, f)
    t1 = time.perf_counter() - t0
    t0 = time.perf_counter(); ensamblar_vectorizado(nd, el, f)
    t2 = time.perf_counter() - t0
    print(f'{nn:5d} {len(nd):7d} {t1*1e3:13.1f} ms {t2*1e3:11.1f} ms {len(nd)**2*8/1e6:14.1f} MB')

La diferencia de tiempo es grande, pero lo que realmente impide usar la versión
densa es la última columna: con `n = 128` la matriz llena pediría más de 2 GB,
aunque solo unas 7 entradas por fila sean distintas de cero.

## 5. Convergencia

Con elementos P1 el error en norma infinito debe comportarse como $h^2$: al
duplicar `n`, el error se divide entre 4.

In [ ]:
ns = [4, 8, 16, 32, 64]
hh, err = [], []

print(f'{"n":>4} {"h":>9} {"error":>13} {"razon":>8}')
for nn in ns:
    nd, el = squaremesh_basico(nn)
    NVn = len(nd)
    libres = np.setdiff1d(np.arange(NVn), nodos_frontera(el))

    An, bn = ensamblar_vectorizado(nd, el, f)
    un = np.zeros(NVn)
    un[libres] = spsolve(An.tocsc()[libres][:, libres], bn[libres])

    e = np.linalg.norm(uex(nd[:, 0], nd[:, 1]) - un, np.inf)
    r = err[-1]/e if err else np.nan
    hh.append(1/nn); err.append(e)
    print(f'{nn:4d} {1/nn:9.5f} {e:13.4e} {r:8.3f}')

hh, err = np.array(hh), np.array(err)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4.5))
ax.loglog(hh, err, 'r-o', label='err')
ax.loglog(hh, hh**2, 'k--', label='$h^2$')
ax.set_xlabel('h'); ax.set_ylabel('error')
ax.legend(); ax.grid(True, which='both', alpha=0.3)
plt.show()

---

## 6. Una coincidencia que no es coincidencia

Sobre **esta** malla en particular —cuadrícula uniforme partida en triángulos
rectángulos, todos con la diagonal en la misma dirección— el método de elementos
finitos y el de diferencias finitas dan **exactamente la misma matriz**.

Las contribuciones que acoplan los dos nodos opuestos por la diagonal de cada
celda se cancelan entre los triángulos que comparten esa diagonal: suman cero.
Lo que queda es el acoplamiento con los cuatro vecinos horizontales y
verticales, con diagonal 4 y vecinos $-1$: la fórmula de 5 puntos de
`MDF_converg.ipynb`.

Es el mismo esquema, llegando por dos caminos distintos. Con triángulos
irregulares, o con la diagonal alternada, deja de pasar.

In [ ]:
from scipy.sparse import diags, kron, identity

nn = 8
nd, el = squaremesh_basico(nn)
Afem, _ = ensamblar_vectorizado(nd, el, f)

# nodos interiores, en el mismo orden que usa el MDF: pos = (i-1)(n-1)+(j-1)
interior = [i*(nn+1) + j for i in range(1, nn) for j in range(1, nn)]
bloque = Afem.toarray()[np.ix_(interior, interior)]

# la matriz de 5 puntos, armada como producto de Kronecker (MDF_v3_converg.m)
m = nn - 1
T = diags([-1, 2, -1], [-1, 0, 1], shape=(m, m))
Amdf = (kron(identity(m), T) + kron(T, identity(m))).toarray()

print('bloque interior del MEF == matriz de 5 puntos del MDF:',
      np.allclose(bloque, Amdf))
print('diferencia maxima:', np.abs(bloque - Amdf).max())